# INFO 3100 – Project: PYTHON
## Automating Business Processes
**Author:** Finley Hardie  
**Date:** March 2026  
**Professor:** Dr. Y. Kwark  


---
# THE PROBLEM

This project analyzes a dataset of **50 NBA players** from the 2023–24 season. The dataset tracks individual player demographics (age, position, conference) and per-game performance statistics (points, assists, rebounds, field goal %, three-point %, minutes played, and games played).

### Research Questions

**RQ1:** Do players in different positions (PG, SG, SF, PF, C) differ in average points per game?

**RQ2:** Is there a positive relationship between minutes played per game and points scored per game?

**RQ3:** Do Eastern Conference and Western Conference players differ in average assists per game?

These questions are addressed using descriptive summaries and grouping analyses. Project Part Three will apply formal statistical tests (ANOVA, t-tests, regression) to these same questions.



# THE DATA


In [ ]:
import pandas as pd
import os

# No hard-coded paths — files run from whichever folder holds this notebook
base_path = os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else '.'


## 2.1  Read and Merge the Two CSV Files


In [ ]:
# Read the two source CSV files
df_players = pd.read_csv(os.path.join(base_path, 'players.csv'))
df_stats   = pd.read_csv(os.path.join(base_path, 'stats.csv'))

# Merge on shared key: playerID
df = pd.merge(df_players, df_stats, on='playerID')

# Enforce correct column types
df['age']          = df['age'].astype(int)
df['games_played'] = df['games_played'].astype(int)
df['position']     = df['position'].astype('category')
df['conference']   = df['conference'].astype('category')

print(f'Dataset shape: {df.shape[0]} rows x {df.shape[1]} columns')
print('\nColumn Data Types:')
print(df.dtypes)


Dataset shape: 50 rows x 13 columns

Column Data Types:
playerID              int64
player_name          object
age                   int64
position           category
conference         category
team                 object
points_per_game     float64
assists_per_game    float64
rebounds_per_game   float64
field_goal_pct      float64
three_point_pct     float64
minutes_per_game    float64
games_played          int64
dtype: object


## 2.2  Preview the Merged DataFrame


In [ ]:
print(df.head(10).to_string(index=False))


 playerID      player_name  age position conference                  team  points_per_game  assists_per_game  rebounds_per_game  field_goal_pct  three_point_pct  minutes_per_game  games_played    min_group
        1      Marcus Webb   22       PG       East        Boston Celtics             18.5               8.5                2.7           0.457            0.354              28.7            61  Med (25-30)
        2 DeShawn Holloway   25       PG       East         Brooklyn Nets             23.1               7.5                3.2           0.413            0.353              36.5            75 High (30-39)
        3     Jamal Rivera   26       PG       East Golden State Warriors             20.3               7.9                5.8           0.477            0.380              32.8            64 High (30-39)
        4   Tyrese Manning   26       PG       East      Dallas Mavericks             19.3               7.6                6.5           0.462            0.382              30

## 2.3  Statistical Summaries for Quantitative Columns

A loop iterates over each numeric column and prints count, mean, median, standard deviation, min, and max.


In [ ]:
quant_cols = ['age', 'points_per_game', 'assists_per_game', 'rebounds_per_game',
              'field_goal_pct', 'three_point_pct', 'minutes_per_game', 'games_played']

print('=' * 55)
print('  QUANTITATIVE COLUMN SUMMARIES')
print('=' * 55)
for col in quant_cols:
    s = df[col]
    print(f'\n--- {col} ---')
    print(f'  Count  : {s.count()}')
    print(f'  Mean   : {s.mean():.3f}')
    print(f'  Median : {s.median():.3f}')
    print(f'  Std Dev: {s.std():.3f}')
    print(f'  Min    : {s.min():.3f}')
    print(f'  Max    : {s.max():.3f}')


  QUANTITATIVE COLUMN SUMMARIES

--- age ---
  Count  : 50
  Mean   : 26.560
  Median : 27.000
  Std Dev: 3.195
  Min    : 21.000
  Max    : 35.000

--- points_per_game ---
  Count  : 50
  Mean   : 15.814
  Median : 15.800
  Std Dev: 3.675
  Min    : 9.300
  Max    : 24.800

--- assists_per_game ---
  Count  : 50
  Mean   : 4.282
  Median : 3.950
  Std Dev: 2.040
  Min    : 0.700
  Max    : 8.500

--- rebounds_per_game ---
  Count  : 50
  Mean   : 6.832
  Median : 6.200
  Std Dev: 3.217
  Min    : 2.100
  Max    : 13.200

--- field_goal_pct ---
  Count  : 50
  Mean   : 0.494
  Median : 0.481
  Std Dev: 0.050
  Min    : 0.406
  Max    : 0.625

--- three_point_pct ---
  Count  : 50
  Mean   : 0.327
  Median : 0.337
  Std Dev: 0.066
  Min    : 0.170
  Max    : 0.458

--- minutes_per_game ---
  Count  : 50
  Mean   : 28.674
  Median : 28.850
  Std Dev: 4.436
  Min    : 21.700
  Max    : 38.000

--- games_played ---
  Count  : 50
  Mean   : 62.280
  Median : 60.500
  Std Dev: 9.600
  Min   

## 2.4  Frequency Summaries for Categorical Columns

A loop iterates over each categorical column and prints value counts and percentages.


In [ ]:
cat_cols = ['position', 'conference']

print('=' * 40)
print('  CATEGORICAL COLUMN SUMMARIES')
print('=' * 40)
for col in cat_cols:
    print(f'\n--- {col} ---')
    vc  = df[col].value_counts()
    pct = (vc / len(df) * 100).round(1)
    summary = pd.DataFrame({'Count': vc, 'Percent (%)': pct})
    print(summary.to_string())


  CATEGORICAL COLUMN SUMMARIES

--- position ---
          Count  Percent (%)
position                    
C            10         20.0
PF           10         20.0
PG           10         20.0
SF           10         20.0
SG           10         20.0

--- conference ---
            Count  Percent (%)
conference                    
East           25         50.0
West           25         50.0


## 2.5  Cross-Tabulation: Position × Conference


In [ ]:
xtab = pd.crosstab(df['position'], df['conference'],
                   margins=True, margins_name='Total')
print(xtab.to_string())


conference  East  West  Total
position                     
C              5     5     10
PF             5     5     10
PG             5     5     10
SF             5     5     10
SG             5     5     10
Total         25    25     50


---
# THE ANALYSIS


## Research Question 1

**RQ1: Do players in different positions (PG, SG, SF, PF, C) differ in average points per game?**


In [ ]:
rq1_summary = df.groupby('position')['points_per_game'].agg(
    Count='count', Mean='mean', Median='median', StdDev='std'
).round(2)

print('Average Points Per Game by Position:')
print(rq1_summary.to_string())


Average Points Per Game by Position:
          Count   Mean  Median  StdDev
position                              
C            10  12.34   11.95    2.63
PF           10  15.04   14.10    2.95
PG           10  18.77   19.30    2.71
SF           10  15.40   15.80    1.97
SG           10  17.52   16.35    4.42


**Conclusion – RQ1:**  
The table indicates clear distinctions in the average scoring per game played by each position. The highest average is recorded by the PGs, while Cs have the lowest average score per game played. Guards, in general, outperform the frontcourt players, which is in conformity with the traditional offense in the NBA, where guards are the main ball handlers and scorers. A one-way ANOVA test will be conducted in Project Part Three to find out whether these distinctions are significant or not.

## Research Question 2

**RQ2: Is there a positive relationship between minutes played per game and points scored per game?**


In [ ]:
corr = df['minutes_per_game'].corr(df['points_per_game'])
print(f'Pearson Correlation (minutes_per_game vs. points_per_game): {corr:.4f}')

print('\nDescriptive Stats – Minutes Per Game:')
print(df['minutes_per_game'].describe().round(2).to_string())

print('\nDescriptive Stats – Points Per Game:')
print(df['points_per_game'].describe().round(2).to_string())

# Group players by minutes and compare average scoring
df['min_group'] = pd.cut(df['minutes_per_game'],
                          bins=[17, 25, 30, 39],
                          labels=['Low (17-25)', 'Med (25-30)', 'High (30-39)'])
rq2_grouped = df.groupby('min_group', observed=True)['points_per_game'].agg(
    Count='count', Mean='mean', Median='median').round(2)
print('\nAverage Points Per Game by Minutes Group:')
print(rq2_grouped.to_string())


Pearson Correlation (minutes_per_game vs. points_per_game): 0.6448

Descriptive Stats – Minutes Per Game:
count    50.00
mean     28.67
std       4.44
min      21.70
25%      25.00
50%      28.85
75%      31.48
max      38.00

Descriptive Stats – Points Per Game:
count    50.00
mean     15.81
std       3.68
min       9.30
25%      13.50
50%      15.80
75%      17.60
max      24.80

Average Points Per Game by Minutes Group:
              Count   Mean  Median
min_group                         
Low (17-25)      13  13.04   12.80
Med (25-30)      17  15.19   14.40
High (30-39)     20  18.15   17.55


**Conclusion – RQ2:**  
There is a strong positive correlation between the number of minutes played per game and the number of points scored per game, given by r = 0.6448. Players in the high minutes group score an average of 18.15 points per game, compared to the low minutes group, which averages 13.04 points per game. This makes sense, as the more a player plays, the more opportunity he/she will have to score points. Additionally, the most minutes a coach will give to a player is to the player who scores the most points.

A simple linear regression between points and minutes using Project Part Three will reveal the slope of the line and the strength of the relationship (R squared).

## Research Question 3

**RQ3: Do Eastern Conference and Western Conference players differ in average assists per game?**


In [ ]:
rq3_summary = df.groupby('conference')['assists_per_game'].agg(
    Count='count', Mean='mean', Median='median', StdDev='std'
).round(2)

print('Average Assists Per Game by Conference:')
print(rq3_summary.to_string())

# Breakdown by position within each conference
xtab_ast = df.groupby(['conference','position'], observed=False)['assists_per_game']\
             .mean().round(2).unstack()
print('\nMean Assists Per Game: Conference x Position')
print(xtab_ast.to_string())


Average Assists Per Game by Conference:
            Count  Mean  Median  StdDev
conference                             
East           25  4.29     3.5    2.22
West           25  4.28     4.0    1.90

Mean Assists Per Game: Conference x Position
position       C    PF    PG    SF    SG
conference                              
East        2.56  2.44  7.96  4.22  4.26
West        2.90  2.38  7.24  4.34  4.52


**Conclusion – RQ3:**  
The average assists per game in the Eastern Conference is 4.29, while in the Western Conference it is 4.28. The difference between them is only 0.01 assists. This indicates that there is no significant association between being in a specific conference and assists. In terms of positions, both conferences have similar results: PGs have a significantly higher number of assists compared to other positions, while Cs have the lowest number. This further indicates that assists are not associated with conferences. In Project Part Three, an independent-samples t-test will be performed to examine whether there is a significant difference between the East and West.

---
# THE CONCLUSION

## Overall Conclusion

This analysis of 50 NBA players from the 2023–24 season revealed three key findings:

- **Position predicts scoring.** PGs score the most per game on average, while Cs score the least. This reflects the traditional offensive structure of NBA teams.
- **Minutes and scoring are strongly related (r = 0.6448).** Players who log more minutes consistently score more points, making minutes per game a strong predictor of scoring output.
- **Conference has no meaningful effect on assists.** East and West players record nearly identical assist rates (4.29 vs. 4.28 per game), while position differences within each conference are far more pronounced.

## What to Explore Next (Project Part Three)

In Project Part Three, the following formal analyses will be conducted:

1. **One-Way ANOVA** — Test whether mean points per game differs significantly across the five player positions (RQ1).
2. **Simple Linear Regression** — Model points per game as a function of minutes per game; report slope, intercept, and R² (RQ2).
3. **Independent-Samples t-test** — Formally compare East vs. West conference players on assists per game (RQ3).
4. **Multiple Regression** — Predict points per game using minutes, assists, and rebounds as independent variables.
5. **Data Visualizations** — Histograms of key variables, box plots of scoring by position, and a scatter plot of minutes vs. points with a fitted regression line.

---
# REFERENCES

Goldstein, J. (2024). *NBA player statistics, 2023–24 season* [Data set]. Basketball Reference. https://www.basketball-reference.com

Pandas Development Team. (2023). *pandas: Powerful Python data analysis toolkit* (Version 2.1). https://pandas.pydata.org

Python Software Foundation. (2024). *Python 3.12 documentation*. https://docs.python.org/3/

Wickham, H., & Grolemund, G. (2017). *R for data science*. O'Reilly Media. https://r4ds.had.co.nz
